In [1]:
# E0: Native-language baseline
# XLM-RoBERTa fine-tuned directly on the low-resource dataset

import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    multilabel_confusion_matrix
)


In [2]:

# ============================================================
# 1. Configuration
# ============================================================

MODEL_NAME = "xlm-roberta-base"

TRAIN_DATA_PATH = "db/goemotions/train.tsv"
VALID_DATA_PATH = "db/goemotions/dev.tsv"
TEST_DATA_PATH = "db/goemotions/test.tsv"
EMOTIONS_PATH = "db/goemotions/emotions.txt"

TEXT_COLUMN = "text"
LABEL_COLUMN = "emotion"

MAX_LENGTH = 140
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
PREDICTION_THRESHOLD = 0.5

RANDOM_SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cpu


In [3]:

# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)



In [4]:

# ============================================================
# 3. Load dataset
# ============================================================

train_df = pd.read_csv(TRAIN_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
train_df = train_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(train_df))
print("\nClass distribution:")
print(train_df[LABEL_COLUMN].value_counts())



Number of examples: 43410

Class distribution:
emotion
27           12823
0             2710
4             1873
15            1857
1             1652
             ...  
6,15,22          1
9,10,19          1
7,10,25          1
7,9,24,25        1
0,1,18           1
Name: count, Length: 711, dtype: int64


In [5]:
test_df = pd.read_csv(TEST_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(test_df))
print("\nClass distribution:")
print(test_df[LABEL_COLUMN].value_counts())


Number of examples: 5427

Class distribution:
emotion
27         1606
0           348
15          260
4           236
10          195
           ... 
1,8           1
6,12          1
1,15,17       1
4,22,26       1
10,12         1
Name: count, Length: 275, dtype: int64


In [6]:
valid_df = pd.read_csv(VALID_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
valid_df = valid_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(valid_df))
print("\nClass distribution:")
print(valid_df[LABEL_COLUMN].value_counts())


Number of examples: 5426

Class distribution:
emotion
27         1592
0           326
15          261
4           258
10          212
           ... 
2,9,22        1
3,26          1
5,11          1
1,3,18        1
0,17,22       1
Name: count, Length: 296, dtype: int64


In [7]:

# ============================================================
# 4. Encode labels
# ============================================================

with open(EMOTIONS_PATH, encoding="utf-8") as emotions_file:
    labels = [line.strip() for line in emotions_file if line.strip()]

label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}
NUM_LABELS = len(labels)

def encode_emotions(emotion_indices):
    encoded = np.zeros(NUM_LABELS, dtype=np.float32)

    for index in str(emotion_indices).split(","):
        encoded[int(index)] = 1.0

    return encoded


for df in [train_df, valid_df, test_df]:
    df["label_id"] = df[LABEL_COLUMN].apply(encode_emotions)

NUM_LABELS = len(labels)

print("\nLabels:")
print(label2id)



Labels:
{'admiration': 0, 'amusement': 1, 'anger': 2, 'annoyance': 3, 'approval': 4, 'caring': 5, 'confusion': 6, 'curiosity': 7, 'desire': 8, 'disappointment': 9, 'disapproval': 10, 'disgust': 11, 'embarrassment': 12, 'excitement': 13, 'fear': 14, 'gratitude': 15, 'grief': 16, 'joy': 17, 'love': 18, 'nervousness': 19, 'optimism': 20, 'pride': 21, 'realization': 22, 'relief': 23, 'remorse': 24, 'sadness': 25, 'surprise': 26, 'neutral': 27}


In [8]:
"""
# ============================================================
# 5. Train / validation / test split
# ============================================================

# First separate out the test set
train_val_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df["label_id"]
)

# Then split the remaining 80% into
# 60% train and 20% validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=train_val_df["label_id"]
)
"""
print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))




Dataset sizes:
Train: 43410
Validation: 5426
Test: 5427


In [9]:

# ============================================================
# 6. Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, timeout=120)



In [10]:
# ============================================================
# 7. PyTorch Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe[TEXT_COLUMN].tolist()
        self.labels = dataframe["label_id"].tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32)
        }


train_dataset = EmotionDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = EmotionDataset(
    valid_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = EmotionDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)


In [11]:

# ============================================================
# 8. DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



In [12]:
# ============================================================
# 9. Model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)

model.to(DEVICE)


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [13]:

# ============================================================
# 10. Optimizer and scheduler
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

total_training_steps = (
    len(train_loader) * EPOCHS
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps
)



In [14]:
# ============================================================
# 11. Evaluation function
# ============================================================

def evaluate(model, dataloader):

    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels_batch = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )

            total_loss += outputs.loss.item()

            predictions = (
                torch.sigmoid(outputs.logits) >= PREDICTION_THRESHOLD
            ).int()

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy().astype(int))

    average_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "labels": all_labels,
        "predictions": all_predictions
    }


In [ ]:

# ============================================================
# 12. Training
# ============================================================

best_val_f1 = -float("inf")
best_model_path = "best_e0_model.pt"

for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_batch = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss

        loss.backward()

        # Prevent excessively large gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    average_train_loss = (
        total_train_loss / len(train_loader)
    )

    # Validation
    val_results = evaluate(
        model,
        val_loader
    )

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"Train loss: "
        f"{average_train_loss:.4f}"
    )

    print(
        f"Validation loss: "
        f"{val_results['loss']:.4f}"
    )

    print(
        f"Validation accuracy: "
        f"{val_results['accuracy']:.4f}"
    )

    print(
        f"Validation macro-F1: "
        f"{val_results['macro_f1']:.4f}"
    )

    print(
        f"Validation weighted-F1: "
        f"{val_results['weighted_f1']:.4f}"
    )

    # Save best model based on macro-F1
    if val_results["macro_f1"] > best_val_f1:

        best_val_f1 = val_results["macro_f1"]

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Saved new best model.")



KeyboardInterrupt: 

: 

In [ ]:

# ============================================================
# 13. Load best model
# ============================================================

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=DEVICE
    )
)



In [ ]:

# ============================================================
# 14. Final test evaluation
# ============================================================

test_results = evaluate(
    model,
    test_loader
)

print("\n==============================")
print("FINAL E0 TEST RESULTS")
print("==============================")

print(
    f"Accuracy:     "
    f"{test_results['accuracy']:.4f}"
)

print(
    f"Macro-F1:     "
    f"{test_results['macro_f1']:.4f}"
)

print(
    f"Weighted-F1:  "
    f"{test_results['weighted_f1']:.4f}"
)



In [ ]:
# ============================================================
# 15. Per-label results
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        test_results["labels"],
        test_results["predictions"],
        target_names=labels,
        digits=4,
        zero_division=0
    )
)


In [ ]:
# ============================================================
# 16. Per-label confusion matrices
# ============================================================

confusion_matrices = multilabel_confusion_matrix(
    test_results["labels"],
    test_results["predictions"]
)

print("\nPer-label Confusion Matrices:")
for label, matrix in zip(labels, confusion_matrices):
    print(f"{label}:\n{matrix}")
